# Datacenter SOC — dual-role training (Defender / Adversary)

This notebook uses the **SOC reward decomposition** from `server/datacenter_env.py`:
**Total** = Outcome + Integrity + Stealth (see `soc_reward_fn`).

Dataset prompts alternate **`[ROLE: DEFENDER]`** and **`[ROLE: ADVERSARY]`** (50/50).
There is **no** chess engine, Stockfish, or `python-chess` in this pipeline.

In [ ]:
# Optional: install training stack in your runtime (e.g. Kaggle). No Stockfish/chess packages.
# %pip install -q torch trl transformers datasets matplotlib

In [ ]:
import os
import random
import sys

# Project root = directory that contains `server/` (upload or clone as `Datacenter-workload_migration/`).
try:
    NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:  # Jupyter / Kaggle
    NOTEBOOK_DIR = os.getcwd()
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

from server.datacenter_env import (
    ADVERSARY_ID,
    DEFENDER_ID,
    soc_reward_fn,
)

# Example: one step’s reward buckets (tune args to match your rollout)
total, outcome, integrity, stealth = soc_reward_fn(
    tier=DEFENDER_ID,
    threat=0.35,
    tool_name="migrate_workload",
    thought="Migrating analytics off the shadow-heavy pod per policy.",
    actual_tags={"analytics", "pci"},
    scans_used=1,
)
print("Total:", total, "| Outcome:", outcome, "| Integrity:", integrity, "| Stealth:", stealth)

In [ ]:
def build_prompt_dataset(n: int = 256, seed: int = 42) -> list[dict]:
    """50/50 DEFENDER vs ADVERSARY system-style prompts for GRPO / SFT prep."""
    rng = random.Random(seed)
    rows: list[dict] = []
    for i in range(n):
        is_def = i % 2 == 0
        role = "[ROLE: DEFENDER]" if is_def else "[ROLE: ADVERSARY]"
        body = (
            f"{role} You are in the Global SOC. Inspect workload placement and routing; "
            f"propose a single authorized migration with structured reasoning. episode={i}"
        )
        rows.append({"idx": i, "prompt": body, "meta": {"side": ("defender" if is_def else "adversary")}})
    rng.shuffle(rows)
    return rows

dataset_preview = build_prompt_dataset(8)
dataset_preview

In [ ]:
def soc_reward_fn_trainer(*, side: str, **kwargs) -> dict[str, float]:
    """Wrap :func:`soc_reward_fn` and return a dict for logging (no chess metrics)."""
    tier = DEFENDER_ID if side == "defender" else ADVERSARY_ID
    total, outcome, integrity, stealth = soc_reward_fn(tier=tier, **kwargs)
    return {
        "total": total,
        "outcome": outcome,
        "integrity": integrity,
        "stealth": stealth,
    }

In [ ]:
import matplotlib.pyplot as plt


def plot_soc_training_curves(history: list[dict]) -> None:
    """Plot Total, Stealth, and Integrity (replaces old white/black, sf_acc)."""
    if not history:
        print("No history to plot.")
        return
    xs = list(range(1, len(history) + 1))
    totals = [h["total"] for h in history]
    stealths = [h["stealth"] for h in history]
    integs = [h["integrity"] for h in history]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(xs, totals, label="Total", color="C0")
    ax.plot(xs, stealths, label="Stealth", color="C1", linestyle="--")
    ax.plot(xs, integs, label="Integrity", color="C2", linestyle=":")
    ax.set_xlabel("Step")
    ax.set_ylabel("Reward component")
    ax.legend()
    ax.set_title("SOC dual-role training — Total / Stealth / Integrity")
    plt.tight_layout()
    plt.show()


demo = [
    soc_reward_fn_trainer(
        side="defender" if i % 2 == 0 else "adversary",
        threat=0.2 + 0.01 * i,
        tool_name="migrate_workload",
        thought="Stabilize tier-1 path; avoid saturation.",
        actual_tags={"api", "web"},
        scans_used=1,
    )
    for i in range(12)
]
plot_soc_training_curves(demo)
